<a href="https://colab.research.google.com/github/Ariiiiii22/MetroCDMX_DFS-y-BFS/blob/main/Lineas_del_metro_hill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math

# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial # Estado inicial
        self.goal = goal # Meta

    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Función de desempeño
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

In [2]:
class Node:
    def __init__(self, state, parent = None, action = None, path_cost = 0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

In [3]:
LINEAS = {
    "1": ["Observatorio", "Tacubaya", "Juanacatlán", "Chapultepec", "Sevilla", "Insurgentes",
          "Cuauhtémoc", "Balderas", "Salto del Agua", "Isabel la Católica", "Pino Suárez", "Merced",
          "Candelaria", "San Lázaro", "Moctezuma", "Balbuena", "Boulevard Puerto Aéreo",
          "Gómez Farías", "Zaragoza", "Pantitlán"],
    "2": ["Cuatro Caminos", "Panteones", "Tacuba", "Cuitláhuac", "Popotla", "Colegio Militar",
          "Normal", "San Cosme", "Revolución", "Hidalgo", "Bellas Artes", "Allende", "Zócalo",
          "Pino Suárez", "San Antonio Abad", "Chabacano", "Viaducto", "Xola", "Villa de Cortés",
          "Nativitas", "Portales", "Ermita", "General Anaya", "Tasqueña"],
    "3": ["Indios Verdes", "Deportivo 18 de Marzo", "Potrero", "La Raza", "Tlatelolco", "Guerrero",
          "Hidalgo", "Juárez", "Balderas", "Niños Héroes", "Hospital General", "Centro Médico",
          "Etiopía", "Eugenia", "División del Norte", "Zapata", "Coyoacán", "Viveros",
          "Miguel Ángel de Quevedo", "Copilco", "Universidad"],
    "4": ["Martín Carrera", "Talismán", "Bondojito", "Consulado", "Canal del Norte", "Morelos",
          "Candelaria", "Fray Servando", "Jamaica", "Santa Anita"],
    "5": ["Pantitlán", "Hangares", "Terminal Aérea", "Oceanía", "Aragón", "Eduardo Molina",
          "Consulado", "Valle Gómez", "Misterios", "La Raza", "Autobuses del Norte",
          "Instituto del Petróleo", "Politécnico"],
    "6": ["El Rosario", "Tezozómoc", "UAM-Azcapotzalco", "Ferrería", "Norte 45", "Vallejo",
          "Instituto del Petróleo", "Lindavista", "Deportivo 18 de Marzo", "La Villa-Basílica",
          "Martín Carrera"],
    "7": ["El Rosario", "Aquiles Serdán", "Camarones", "Refinería", "Tacuba", "San Joaquín",
          "Polanco", "Auditorio", "Constituyentes", "Tacubaya", "San Pedro de los Pinos",
          "San Antonio", "Mixcoac", "Barranca del Muerto"],
    "8": ["Garibaldi", "Bellas Artes", "San Juan de Letrán", "Salto del Agua", "Doctores", "Obrera",
          "Chabacano", "La Viga", "Santa Anita", "Coyuya", "Iztacalco", "Apatlaco", "Aculco",
          "Escuadrón 201", "Atlalilco", "Iztapalapa", "Cerro de la Estrella", "UAM-I",
          "Constitución de 1917"],
    "9": ["Tacubaya", "Patriotismo", "Chilpancingo", "Centro Médico", "Lázaro Cárdenas", "Chabacano",
          "Jamaica", "Mixiuhca", "Velódromo", "Ciudad Deportiva", "Puebla", "Pantitlán"],
    "A": ["Pantitlán", "Agrícola Oriental", "Canal de San Juan", "Tepalcates", "Guelatao",
          "Peñón Viejo", "Acatitla", "Santa Marta", "Los Reyes", "La Paz"],
    "B": ["Buenavista", "Guerrero", "Garibaldi", "Lagunilla", "Tepito", "Morelos", "San Lázaro",
          "Ricardo Flores Magón", "Romero Rubio", "Oceanía", "Deportivo Oceanía", "Bosque de Aragón",
          "Villa de Aragón", "Nezahualcóyotl", "Impulsora", "Río de los Remedios", "Múzquiz",
          "Ecatepec", "Olímpica", "Plaza Aragón", "Ciudad Azteca"],
    "12": ["Mixcoac", "Insurgentes Sur", "Hospital 20 de Noviembre", "Zapata", "Parque de los Venados",
           "Eje Central", "Ermita", "Mexicaltzingo", "Atlalilco", "Culhuacán", "San Andrés Tomatlán",
           "Lomas Estrella", "Calle 11", "Periférico Oriente", "Tezonco", "Olivos", "Nopalera",
           "Zapotitlán", "Tlaltenco", "Tláhuac"],
}

In [4]:
def construir_metro(lineas):
    """Devuelve un dict con el mismo formato que romania: {estacion: {vecino: costo}}."""
    grafo = {}
    for estaciones in lineas.values():
        for e in estaciones:
            grafo.setdefault(e, {})
        for a, b in zip(estaciones, estaciones[1:]):
            grafo[a][b] = 1          # costo unitario
            grafo[b][a] = 1
    return grafo

metro = construir_metro(LINEAS)
print("Estaciones:", len(metro))
print("Tramos:", sum(len(v) for v in metro.values()) // 2)

Estaciones: 163
Tramos: 183


In [5]:
COORDENADAS = {
    "Observatorio": (19.3986, -99.2008),
    "Tacubaya": (19.4026, -99.1874),
    "Juanacatlán": (19.4117, -99.1818),
    "Chapultepec": (19.4208, -99.1763),
    "Sevilla": (19.4220, -99.1695),
    "Insurgentes": (19.4231, -99.1627),
    "Cuauhtémoc": (19.4252, -99.1559),
    "Balderas": (19.4272, -99.1490),
    "Salto del Agua": (19.4263, -99.1424),
    "Isabel la Católica": (19.4260, -99.1377),
    "Pino Suárez": (19.4256, -99.1330),
    "Merced": (19.4270, -99.1263),
    "Candelaria": (19.4284, -99.1196),
    "San Lázaro": (19.4301, -99.1146),
    "Moctezuma": (19.4265, -99.1081),
    "Balbuena": (19.4229, -99.1016),
    "Boulevard Puerto Aéreo": (19.4193, -99.0952),
    "Gómez Farías": (19.4157, -99.0887),
    "Zaragoza": (19.4121, -99.0822),
    "Pantitlán": (19.4150, -99.0720),
    "Cuatro Caminos": (19.4592, -99.2147),
    "Panteones": (19.4590, -99.2015),
    "Tacuba": (19.4587, -99.1883),
    "Cuitláhuac": (19.4556, -99.1829),
    "Popotla": (19.4524, -99.1775),
    "Colegio Militar": (19.4493, -99.1721),
    "Normal": (19.4462, -99.1667),
    "San Cosme": (19.4430, -99.1613),
    "Revolución": (19.4399, -99.1559),
    "Hidalgo": (19.4366, -99.1470),
    "Bellas Artes": (19.4363, -99.1413),
    "Allende": (19.4344, -99.1372),
    "Zócalo": (19.4326, -99.1332),
    "San Antonio Abad": (19.4171, -99.1342),
    "Chabacano": (19.4087, -99.1355),
    "Viaducto": (19.4018, -99.1372),
    "Xola": (19.3950, -99.1390),
    "Villa de Cortés": (19.3865, -99.1397),
    "Nativitas": (19.3780, -99.1404),
    "Portales": (19.3695, -99.1411),
    "Ermita": (19.3626, -99.1428),
    "General Anaya": (19.3533, -99.1416),
    "Tasqueña": (19.3440, -99.1405),
    "Indios Verdes": (19.4947, -99.1198),
    "Deportivo 18 de Marzo": (19.4853, -99.1258),
    "Potrero": (19.4772, -99.1317),
    "La Raza": (19.4690, -99.1375),
    "Tlatelolco": (19.4569, -99.1415),
    "Guerrero": (19.4448, -99.1456),
    "Juárez": (19.4326, -99.1479),
    "Niños Héroes": (19.4196, -99.1517),
    "Hospital General": (19.4121, -99.1544),
    "Centro Médico": (19.4066, -99.1546),
    "Etiopía": (19.3985, -99.1560),
    "Eugenia": (19.3904, -99.1573),
    "División del Norte": (19.3823, -99.1587),
    "Zapata": (19.3705, -99.1650),
    "Coyoacán": (19.3600, -99.1703),
    "Viveros": (19.3535, -99.1750),
    "Miguel Ángel de Quevedo": (19.3438, -99.1747),
    "Copilco": (19.3341, -99.1744),
    "Universidad": (19.3244, -99.1741),
    "Martín Carrera": (19.4846, -99.1062),
    "Talismán": (19.4767, -99.1083),
    "Bondojito": (19.4689, -99.1104),
    "Consulado": (19.4610, -99.1125),
    "Canal del Norte": (19.4516, -99.1155),
    "Morelos": (19.4422, -99.1186),
    "Fray Servando": (19.4188, -99.1207),
    "Jamaica": (19.4093, -99.1218),
    "Santa Anita": (19.4022, -99.1215),
    "Hangares": (19.4252, -99.0773),
    "Terminal Aérea": (19.4353, -99.0827),
    "Oceanía": (19.4455, -99.0880),
    "Aragón": (19.4507, -99.0962),
    "Eduardo Molina": (19.4558, -99.1043),
    "Valle Gómez": (19.4637, -99.1208),
    "Misterios": (19.4663, -99.1292),
    "Autobuses del Norte": (19.4800, -99.1400),
    "Instituto del Petróleo": (19.4890, -99.1450),
    "Politécnico": (19.5000, -99.1496),
    "El Rosario": (19.5046, -99.2000),
    "Tezozómoc": (19.5020, -99.1908),
    "UAM-Azcapotzalco": (19.4994, -99.1817),
    "Ferrería": (19.4968, -99.1725),
    "Norte 45": (19.4942, -99.1633),
    "Vallejo": (19.4916, -99.1542),
    "Lindavista": (19.4866, -99.1342),
    "La Villa-Basílica": (19.4843, -99.1176),
    "Aquiles Serdán": (19.4931, -99.1971),
    "Camarones": (19.4817, -99.1942),
    "Refinería": (19.4702, -99.1912),
    "San Joaquín": (19.4475, -99.1881),
    "Polanco": (19.4363, -99.1879),
    "Auditorio": (19.4250, -99.1878),
    "Constituyentes": (19.4138, -99.1876),
    "San Pedro de los Pinos": (19.3937, -99.1873),
    "San Antonio": (19.3849, -99.1871),
    "Mixcoac": (19.3760, -99.1870),
    "Barranca del Muerto": (19.3612, -99.1895),
    "Garibaldi": (19.4425, -99.1397),
    "San Juan de Letrán": (19.4313, -99.1419),
    "Doctores": (19.4204, -99.1401),
    "Obrera": (19.4146, -99.1378),
    "La Viga": (19.4055, -99.1285),
    "Coyuya": (19.3945, -99.1179),
    "Iztacalco": (19.3868, -99.1142),
    "Apatlaco": (19.3791, -99.1105),
    "Aculco": (19.3714, -99.1069),
    "Escuadrón 201": (19.3637, -99.1033),
    "Atlalilco": (19.3560, -99.0996),
    "Iztapalapa": (19.3563, -99.0909),
    "Cerro de la Estrella": (19.3567, -99.0823),
    "UAM-I": (19.3570, -99.0737),
    "Constitución de 1917": (19.3573, -99.0650),
    "Patriotismo": (19.4039, -99.1765),
    "Chilpancingo": (19.4053, -99.1655),
    "Lázaro Cárdenas": (19.4087, -99.1450),
    "Mixiuhca": (19.4104, -99.1118),
    "Velódromo": (19.4116, -99.1019),
    "Ciudad Deportiva": (19.4127, -99.0919),
    "Puebla": (19.4139, -99.0820),
    "Agrícola Oriental": (19.4083, -99.0597),
    "Canal de San Juan": (19.4017, -99.0473),
    "Tepalcates": (19.3950, -99.0350),
    "Guelatao": (19.3883, -99.0227),
    "Peñón Viejo": (19.3817, -99.0103),
    "Acatitla": (19.3750, -98.9980),
    "Santa Marta": (19.3683, -98.9857),
    "Los Reyes": (19.3617, -98.9733),
    "La Paz": (19.3550, -98.9610),
    "Buenavista": (19.4466, -99.1525),
    "Lagunilla": (19.4424, -99.1327),
    "Tepito": (19.4423, -99.1256),
    "Ricardo Flores Magón": (19.4352, -99.1057),
    "Romero Rubio": (19.4404, -99.0969),
    "Deportivo Oceanía": (19.4535, -99.0825),
    "Bosque de Aragón": (19.4616, -99.0769),
    "Villa de Aragón": (19.4696, -99.0714),
    "Nezahualcóyotl": (19.4777, -99.0658),
    "Impulsora": (19.4857, -99.0603),
    "Río de los Remedios": (19.4938, -99.0547),
    "Múzquiz": (19.5018, -99.0492),
    "Ecatepec": (19.5099, -99.0436),
    "Olímpica": (19.5179, -99.0381),
    "Plaza Aragón": (19.5260, -99.0325),
    "Ciudad Azteca": (19.5340, -99.0270),
    "Insurgentes Sur": (19.3742, -99.1797),
    "Hospital 20 de Noviembre": (19.3723, -99.1723),
    "Parque de los Venados": (19.3679, -99.1576),
    "Eje Central": (19.3652, -99.1502),
    "Mexicaltzingo": (19.3593, -99.1212),
    "Culhuacán": (19.3497, -99.0916),
    "San Andrés Tomatlán": (19.3435, -99.0837),
    "Lomas Estrella": (19.3372, -99.0757),
    "Calle 11": (19.3309, -99.0677),
    "Periférico Oriente": (19.3246, -99.0598),
    "Tezonco": (19.3184, -99.0518),
    "Olivos": (19.3121, -99.0439),
    "Nopalera": (19.3058, -99.0359),
    "Zapotitlán": (19.2995, -99.0279),
    "Tlaltenco": (19.2933, -99.0200),
    "Tláhuac": (19.2870, -99.0120),
}

In [6]:
def distancia_km(est1, est2):
    """Distancia en línea recta (fórmula de Haversine) entre dos estaciones, en km."""
    lat1, lon1 = COORDENADAS[est1]
    lat2, lon2 = COORDENADAS[est2]
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

print(f"Cuatro Caminos -> Pantitlán en línea recta: {distancia_km('Cuatro Caminos', 'Pantitlán'):.2f} km")

Cuatro Caminos -> Pantitlán en línea recta: 15.75 km


In [7]:
class GraphHillClimbingProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    # Función de transición
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

    # Heurística: distancia en línea recta de la estación a la meta
    def h(self, state):
        return distancia_km(state, self.goal)

In [8]:
def hill_climbing(problem):
    current = Node(problem.initial)

    while True:
        if problem.is_goal(current.state):
            return current

        children = current.expand(problem)
        if not children:
            return current

        # El vecino más prometedor: el de menor h
        best = min(children, key=lambda n: problem.h(n.state))

        # Si ningún vecino mejora al actual, estamos atascados (óptimo local)
        if problem.h(best.state) >= problem.h(current.state):
            return current

        current = best

In [10]:
rutas = [("Cuatro Caminos", "Pantitlán"),
         ("Politécnico", "Tasqueña"),
         ("Zapata", "Oceanía")]

resultados = {}
for inicio, meta in rutas:
    problem = GraphHillClimbingProblem(inicio, meta, metro)
    node = hill_climbing(problem)
    llego = problem.is_goal(node.state)
    resultados[(inicio, meta)] = (node, llego)

    print("=" * 70)
    print(f"{inicio} -> {meta}")
    print("=" * 70)
    print("Ruta:", " -> ".join(node.path()))
    if llego:
        print(f"Llegó a la meta en {int(node.path_cost)} pasos.")
    else:
        print(f"ATASCADO en '{node.state}' (h = {problem.h(node.state):.2f} km de la meta), "
              f"después de {int(node.path_cost)} km.")
    print()

Cuatro Caminos -> Pantitlán
Ruta: Cuatro Caminos -> Panteones -> Tacuba -> Cuitláhuac -> Popotla -> Colegio Militar -> Normal -> San Cosme -> Revolución -> Hidalgo -> Bellas Artes -> Allende -> Zócalo -> Pino Suárez -> Merced -> Candelaria -> San Lázaro -> Moctezuma -> Balbuena -> Boulevard Puerto Aéreo -> Gómez Farías -> Zaragoza -> Pantitlán
Llegó a la meta en 22 pasos.

Politécnico -> Tasqueña
Ruta: Politécnico -> Instituto del Petróleo -> Autobuses del Norte -> La Raza -> Tlatelolco -> Guerrero -> Hidalgo -> Juárez -> Balderas -> Niños Héroes -> Hospital General -> Centro Médico -> Etiopía -> Eugenia -> División del Norte -> Zapata -> Parque de los Venados -> Eje Central -> Ermita -> General Anaya -> Tasqueña
Llegó a la meta en 20 pasos.

Zapata -> Oceanía
Ruta: Zapata -> División del Norte -> Eugenia -> Etiopía -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Fray Servando -> Candelaria -> Morelos -> Canal del Norte
ATASCADO en 'Canal del Norte' (h = 2.96 km de la me